In [ ]:
%reload_ext autoreload
%autoreload 2


import numpy as np
import sympy as sym
from IPython.display import Math, display
from scipy.linalg import null_space

import config  # noqa: F401  (side effect: puts the project root on sys.path)
from modules import *
from modules.imu_kinematics_sym import *
from modules.speaker_kinematics_sym import *

sym.init_printing(use_unicode=True)

# Gravity prediction sign inverted

In [2]:
gravity_sym = sym.Symbol(r"g", real=True, positive=True)
gravity_vector_sym = sym.Matrix([0, 0, -1 * gravity_sym])

q = np.array([0.9996, 0.02769, 0.01076, -0.0000006])
g_h = -R(q).T @ np.array([0, 0, -9.81])
g_h

array([-0.21102708,  0.54306036,  9.79268508])

In [3]:
g_h_old = R(q).T @ np.array([0, 0, -9.81])
print(f"{g_h_old} <- aktuell")

g_h_no_transpose = R(q) @ np.array([0, 0, -9.81])
print(f"{g_h_no_transpose} <- kein transpose")

g_h_sign_reversed = -R(q).T @ np.array([0, 0, -9.81])
print(f"{g_h_sign_reversed} <- -R(q)^T")

g_h_both = -R(q) @ np.array([0, 0, -9.81])
print(f"{g_h_both} <- beides")

[ 0.21102708 -0.54306036 -9.79268508] <- aktuell
[-0.21102643  0.54306062 -9.79268508] <- kein transpose
[-0.21102708  0.54306036  9.79268508] <- -R(q)^T
[ 0.21102643 -0.54306062  9.79268508] <- beides


# Constructing Matrices

## The main two variables that determine Matrix size/layout/content

1. number of speakers $N$

2. which measurement $y$? ($y_{g}, y_{p_{s}q_{s}}, y_{p}, y_{\psi}$)

### 1. number of speakers

Extract it from the code using the length of the state `self.x` of the Filter class instance.

```
N = (len(self.x) - get_def("imu nominal state")["len"]) / get_def("speaker nominal state")["len"]
```